# Financial Crime Customer Risk & KYC/EDD Analytics
Synthetic portfolio analysis.

In [ ]:
import pandas as pd
customers = pd.read_csv('../data/processed/customer_risk_analytics.csv')
cases = pd.read_csv('../data/raw/kyc_review_cases.csv')
customers.head()


## Risk Tier Distribution

In [ ]:
customers['risk_level'].value_counts()


## KYC Completeness

In [ ]:
customers.groupby('customer_type')['kyc_complete_pct'].mean()


## EDD Drivers

In [ ]:
customers[customers.edd_required==1]['primary_risk_driver'].value_counts()


## Review Case Aging

In [ ]:
cases[['case_status','age_days','risk_score']].describe(include='all')


## EDA + Feature Engineering

This section extends the existing **Customer Risk & KYC/EDD** analysis with a simple, practical workflow.

The purpose is to show that the project also includes:
- dataset review and data-quality checks,
- missing and duplicate validation,
- summary statistics and basic outlier review,
- simple KYC/EDD feature engineering,
- business-rule checks,
- and an analysis-ready dataset.

The existing risk-tier, KYC completeness, EDD-driver, and case-aging analysis above is kept unchanged.


### 1. Dataset Review & Data Quality

In [ ]:
# Review both datasets already loaded in the notebook
print("CUSTOMER DATA")
print("Shape:", customers.shape)
display(customers.head())
display(customers.dtypes.to_frame("data_type"))

print("\nKYC REVIEW CASE DATA")
print("Shape:", cases.shape)
display(cases.head())
display(cases.dtypes.to_frame("data_type"))


In [ ]:
# Missing values and duplicate validation
customer_quality = pd.DataFrame({
    "missing_values": customers.isna().sum(),
    "missing_pct": (customers.isna().mean() * 100).round(2)
}).sort_values("missing_values", ascending=False)

case_quality = pd.DataFrame({
    "missing_values": cases.isna().sum(),
    "missing_pct": (cases.isna().mean() * 100).round(2)
}).sort_values("missing_values", ascending=False)

print("Customer duplicates:", customers.duplicated().sum())
display(customer_quality)

print("Case duplicates:", cases.duplicated().sum())
display(case_quality)


### 2. Basic EDA & Range Checks

In [ ]:
# Basic summary statistics for important fields already used in the project
customer_fields = [
    c for c in ['kyc_complete_pct']
    if c in customers.columns
]

case_fields = [
    c for c in ['age_days', 'risk_score']
    if c in cases.columns
]

if customer_fields:
    print("Customer summary")
    display(customers[customer_fields].describe().T)

if case_fields:
    print("KYC review case summary")
    display(cases[case_fields].describe().T)

# Simple range validation
if 'kyc_complete_pct' in customers.columns:
    invalid_kyc = customers[
        (customers['kyc_complete_pct'] < 0) |
        (customers['kyc_complete_pct'] > 100)
    ]
    print("Invalid KYC completeness records:", len(invalid_kyc))

if 'age_days' in cases.columns:
    print("Negative case-age records:", (cases['age_days'] < 0).sum())

if 'risk_score' in cases.columns:
    print("Missing risk-score records:", cases['risk_score'].isna().sum())


In [ ]:
# Simple outlier review using the IQR method
# We flag unusual values for review rather than automatically removing them.
outlier_rows = []

for dataset_name, df, fields in [
    ("customers", customers, ['kyc_complete_pct']),
    ("cases", cases, ['age_days', 'risk_score'])
]:
    for col in fields:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1

            if iqr > 0:
                lower = q1 - (1.5 * iqr)
                upper = q3 + (1.5 * iqr)
                count = ((df[col] < lower) | (df[col] > upper)).sum()

                outlier_rows.append({
                    "dataset": dataset_name,
                    "field": col,
                    "potential_outliers": int(count)
                })

display(pd.DataFrame(outlier_rows))


### 3. Feature Engineering

The features below are intentionally simple and directly related to the KYC/EDD workflow.


In [ ]:
# Work on copies so the original analysis above stays unchanged
customers_fe = customers.copy()
cases_fe = cases.copy()

created_customer_features = []
created_case_features = []

# 1. KYC documentation gap
if 'kyc_complete_pct' in customers_fe.columns:
    customers_fe['kyc_gap_pct'] = (100 - customers_fe['kyc_complete_pct']).clip(lower=0)
    created_customer_features.append('kyc_gap_pct')

    customers_fe['kyc_completion_status'] = customers_fe['kyc_complete_pct'].apply(
        lambda x: 'Complete' if x >= 100 else 'Incomplete'
    )
    created_customer_features.append('kyc_completion_status')

# 2. Combined EDD/KYC attention flag
if {'edd_required', 'kyc_complete_pct'}.issubset(customers_fe.columns):
    customers_fe['edd_attention_flag'] = (
        (customers_fe['edd_required'] == 1) |
        (customers_fe['kyc_complete_pct'] < 100)
    ).astype(int)
    created_customer_features.append('edd_attention_flag')

# 3. Review-age bucket
if 'age_days' in cases_fe.columns:
    cases_fe['case_age_bucket'] = pd.cut(
        cases_fe['age_days'],
        bins=[-1, 7, 30, 60, float('inf')],
        labels=['0-7 days', '8-30 days', '31-60 days', '60+ days']
    )
    created_case_features.append('case_age_bucket')

    # A simple portfolio flag for cases older than 30 days.
    # Change this threshold if your organization's policy is different.
    cases_fe['aged_case_flag'] = (cases_fe['age_days'] > 30).astype(int)
    created_case_features.append('aged_case_flag')

print("Customer features created:", created_customer_features)
print("Case features created:", created_case_features)

display(customers_fe.head())
display(cases_fe.head())


### 4. Business Rule & KPI Validation

In [ ]:
# Check whether engineered flags behave as expected
if 'edd_attention_flag' in customers_fe.columns:
    print("Customers requiring EDD/KYC attention:")
    display(customers_fe['edd_attention_flag'].value_counts(dropna=False))

if 'case_age_bucket' in cases_fe.columns:
    print("KYC review case aging:")
    display(cases_fe['case_age_bucket'].value_counts(dropna=False).sort_index())

# Reconfirm the main KPIs already used in the project
if 'risk_level' in customers_fe.columns:
    print("Risk tier distribution:")
    display(customers_fe['risk_level'].value_counts(dropna=False))

if 'edd_required' in customers_fe.columns:
    print("EDD-required rate:",
          round(customers_fe['edd_required'].mean() * 100, 2), "%")


### 5. Final Validation & Analysis-Ready Output

In [ ]:
print("Final customer dataset shape:", customers_fe.shape)
print("Final case dataset shape:", cases_fe.shape)
print("Customer duplicate rows:", customers_fe.duplicated().sum())
print("Case duplicate rows:", cases_fe.duplicated().sum())

# Optional exports for Tableau / Streamlit / further analysis.
# Uncomment when you want to save the enriched datasets.
#
# customers_fe.to_csv(
#     '../data/processed/customer_risk_analytics_enriched.csv',
#     index=False
# )
#
# cases_fe.to_csv(
#     '../data/processed/kyc_review_cases_enriched.csv',
#     index=False
# )

print("EDA + Feature Engineering completed.")
